In [1]:
import gc

import numpy as np
import jax
from jax import numpy as jnp
import equinox as eqx
import optax
from tqdm import tqdm

from functools import partial

from jepax.data import build_dataloader
from jepax.model import get_ijepa_model, IJEPAEncoder

key = jax.random.key(0)

encoder = IJEPAEncoder(
    key=key,
    num_channels=3,
    patch_size=2,
    dim=12,
    num_head=3,
    num_layers=6,
    img_size=32
)

/Users/anton/source/jepax/jepax/model/transformer.py:227: UserWarning: A JAX array is being set as static! This can result in unexpected behavior and is usually a mistake to do.
  self.pe = PositionalEncoding2D(


In [8]:

def init_layernorm(model):
    is_layernorm = lambda x: isinstance(x, eqx.nn.LayerNorm)

    def init_fn(dim):
        return 1.0 * jnp.ones(dim), jnp.zeros(dim)
    
    print(jax.tree_util.tree_leaves(model, is_leaf=is_layernorm))

    get_params = lambda m: [
        attr
        for x in jax.tree_util.tree_leaves(m, is_leaf=is_layernorm) 
        if is_layernorm(x)
        for attr in (x.weight, x.bias)
    ]

    params = get_params(model)

    init_params = [init_fn(x[0].shape) for x in params]

    new_model = eqx.tree_at(get_params, model, init_params)

    return new_model

init_layernorm(encoder)

[Array([[-1.25446677e-01,  1.66333035e-01, -5.91302030e-02,
         2.44578972e-01, -1.15349978e-01, -2.65297502e-01,
        -1.77476153e-01,  4.80051078e-02, -1.47136286e-01,
        -1.66046858e-01,  6.08824305e-02,  9.69062522e-02],
       [-1.85475752e-01,  2.40808293e-01,  7.86160827e-02,
         2.78045028e-01, -3.26613560e-02,  2.29258746e-01,
         2.65252978e-01, -6.33772882e-03,  8.90074298e-02,
        -1.45136565e-01,  3.82371172e-02, -2.74468780e-01],
       [-7.04799443e-02,  2.35862359e-01, -6.56922981e-02,
         3.08631491e-02, -3.15355770e-02, -7.31674433e-02,
         1.72224969e-01,  2.39055723e-01, -1.77814439e-01,
        -1.90115958e-01,  2.51701653e-01, -1.22168660e-01],
       [ 3.57106701e-02,  4.67330739e-02,  5.52010201e-02,
        -1.04720769e-02, -9.63126346e-02, -2.34869957e-01,
         2.34799281e-01,  2.60789037e-01, -4.77965660e-02,
        -3.45486179e-02,  1.25547856e-01,  1.18038714e-01],
       [-8.64462182e-03, -2.21942872e-01,  1.065713

IJEPAEncoder(
  embed=PatchEmbedding(
    linear=Linear(
      weight=f32[12,12],
      bias=f32[12],
      in_features=12,
      out_features=12,
      use_bias=True
    ),
    patch_size=2
  ),
  transformer=Transformer(
    blocks=[
      TransformerBlock(
        attn=Attention(
          qkv_proj=Linear(
            weight=f32[36,12],
            bias=f32[36],
            in_features=12,
            out_features=36,
            use_bias=True
          ),
          out_proj=Linear(
            weight=f32[12,12],
            bias=f32[12],
            in_features=12,
            out_features=12,
            use_bias=True
          ),
          num_head=3,
          dim=12,
          causal=False
        ),
        ff=FeedForward(
          linear1=Linear(
            weight=f32[36,12],
            bias=f32[36],
            in_features=12,
            out_features=36,
            use_bias=True
          ),
          linear2=Linear(
            weight=f32[12,36],
            bias=f32[1